# HW4: Word2Vec и Retrieval на русском корпусе

In [3]:
import re
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from pymorphy3 import MorphAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

import gensim.downloader as api
from gensim.models import Word2Vec

## 1. Готовая русская модель и препроцессинг

Для задания беру готовую модель `word2vec-ruscorpora-300`.
У неё **не обычный словарь словоформ**, а токены вида:

```text
лемма_POS
```

Следовательно, этапы преобразования:
- приведение к нижнему регистру;
- нормализация `ё -> е`;
- лемматизация;
- добавление POS-тега.

In [4]:
pretrained = api.load("word2vec-ruscorpora-300")
print("Размер словаря:", len(pretrained))
print("Размерность:", pretrained.vector_size)
print("Первые токены:", pretrained.index_to_key[:10])

Размер словаря: 184973
Размерность: 300
Первые токены: ['весь_DET', 'человек_NOUN', 'мочь_VERB', 'год_NOUN', 'сказать_VERB', 'время_NOUN', 'говорить_VERB', 'становиться_VERB', 'знать_VERB', 'самый_DET']


In [5]:
morph = MorphAnalyzer()
TOKEN_RE = re.compile(r"[А-Яа-яЁёA-Za-z]+")

STOPWORDS = {
    'и','в','во','не','что','он','на','я','с','со','как','а','то','все','она','так','его','но','да','ты','к','у','же','вы','за','бы','по','только','ее','мне','было','вот','от','меня','еще','нет','о','из','ему','теперь','когда','даже','ну','вдруг','ли','если','уже','или','ни','быть','был','него','до','вас','нибудь','опять','уж','вам','ведь','там','потом','себя','ничего','ей','может','они','тут','где','есть','надо','ней','для','мы','тебя','их','чем','была','сам','чтоб','без','будто','чего','раз','тоже','себе','под','будет','ж','тогда','кто','этот','того','потому','этого','какой','совсем','ним','здесь','этом','один','почти','мой','тем','чтобы','нее','кажется','сейчас','были','куда','зачем','всех','никогда','можно','при','наконец','два','об','другой','хоть','после','над','больше','тот','через','эти','нас','про','них','какая','много','разве','три','эту','моя','впрочем','хорошо','свою','этой','перед','иногда','лучше','чуть','том','нельзя','такой','им','более','всегда','конечно','всю','между'
}

P2G = {
    'NOUN': 'NOUN',
    'ADJF': 'ADJ',
    'ADJS': 'ADJ',
    'COMP': 'ADJ',
    'VERB': 'VERB',
    'INFN': 'VERB',
    'PRTF': 'VERB',
    'PRTS': 'VERB',
    'GRND': 'VERB',
    'NUMR': 'NUM',
    'ADVB': 'ADV',
    'NPRO': 'PRON',
    'PRED': 'ADV',
    'PREP': 'ADP',
    'CONJ': 'CCONJ',
    'PRCL': 'PART',
    'INTJ': 'INTJ',
}

@lru_cache(maxsize=200_000)
def lemmatize_word(word: str) -> str:
    return morph.parse(word)[0].normal_form


def tokenize(text: str) -> list[str]:
    text = text.lower().replace('ё', 'е')
    words = TOKEN_RE.findall(text)
    lemmas = [lemmatize_word(word) for word in words]
    return [lemma for lemma in lemmas if len(lemma) > 2 and lemma not in STOPWORDS]


def to_ruscorpora_token(word: str) -> str | None:
    word = word.lower().replace('ё', 'е')
    lemma = lemmatize_word(word)
    candidates = []

    for parse in morph.parse(word)[:5]:
        pos = P2G.get(parse.tag.POS)
        if pos is None:
            continue
        token = f"{parse.normal_form}_{pos}"
        if token not in candidates:
            candidates.append(token)

    for fallback_pos in ['NOUN', 'ADJ', 'VERB', 'ADV']:
        token = f"{lemma}_{fallback_pos}"
        if token not in candidates:
            candidates.append(token)

    for token in candidates:
        if token in pretrained:
            return token
    return None

In [6]:
examples = []
for raw_word in ['Москва', 'фильм', 'рубль', 'президент', 'хороший']:
    normalized = raw_word.lower().replace('ё', 'е')
    model_token = to_ruscorpora_token(raw_word)
    examples.append({
        'raw_word': raw_word,
        'raw_in_model': normalized in pretrained,
        'ruscorpora_token': model_token,
        'token_in_model': model_token in pretrained if model_token else False,
    })

preprocess_examples_df = pd.DataFrame(examples)
display(preprocess_examples_df)

,raw_word,raw_in_model,ruscorpora_token,token_in_model
0,Москва,False,москва_NOUN,True
1,фильм,False,фильм_NOUN,True
2,рубль,False,рубль_NOUN,True
3,президент,False,президент_NOUN,True
4,хороший,False,хороший_ADJ,True


In [7]:
selected_words = ['президент', 'компания', 'матч', 'театр', 'рубль', 'министр', 'команда', 'фильм']

rows = []
for word in selected_words:
    token = to_ruscorpora_token(word)
    neighbors = [w for w, _ in pretrained.most_similar(token, topn=10)] if token else []
    rows.append({
        'word': word,
        'ruscorpora_token': token,
        'nearest_neighbors': ', '.join(neighbors),
    })

pretrained_neighbors_df = pd.DataFrame(rows)
display(pretrained_neighbors_df)

,word,ruscorpora_token,nearest_neighbors
0,президент,президент_NOUN,"путин_NOUN, президентский_ADJ, ельцин_NOUN, пр..."
1,компания,компания_NOUN,"фирма_NOUN, holding_NOUN, корпорация_NOUN, exx..."
2,матч,матч_NOUN,"чемпионат_NOUN, турнир_NOUN, сборная_NOUN, цск..."
3,театр,театр_NOUN,"спектакль_NOUN, труппа_NOUN, театральный_ADJ, ..."
4,рубль,рубль_NOUN,"доллар_NOUN, руб_NOUN, копейка_NOUN, целковый_..."
5,министр,министр_NOUN,"министерство_NOUN, финансы_NOUN, статс-секрета..."
6,команда,команда_NOUN,"сборная_NOUN, шамиль::тарпищев_NOUN, кубок::ка..."
7,фильм,фильм_NOUN,"кинофильм_NOUN, кино_NOUN, сериал_NOUN, телефи..."


Наблюдения по готовой модели:
1. Модель действительно ожидает не сырые словоформы, а формат `лемма_POS`. Например, `москва` в словаре нет, а `москва_NOUN` есть.
2. После правильного препроцессинга соседи получаются семантически адекватными: у `театр` появляются `спектакль`, `труппа`, `мхат`, у `рубль` — `доллар`, `копейка`, `сумма`.

## 2. Собственный датасет и обучение Word2Vec

Для собственного корпуса был взят `zloelias/lenta-ru-short` из Hugging Face.

In [8]:
dataset = load_dataset('zloelias/lenta-ru-short', split='train')
documents = [f"{row['title']}. {row['text']}" for row in dataset]
doc_topics = [row['topic'] for row in dataset]

dataset_summary = pd.DataFrame({
    'num_documents': [len(documents)],
    'avg_topic_count_top5': [pd.Series(doc_topics).value_counts().head(5).to_dict()],
})

display(dataset_summary)
print('Пример документа:')
print(documents[0][:500], '...')

,num_documents,avg_topic_count_top5
0,4039,"{'Спорт': 1278, 'Культура': 1152, 'Экономика':..."


Пример документа:
Торговля в США начинает оживляться. Согласно данным министерства торговли США, в июле товарные запасы американских оптовиков снизились на 0,7 процента, что является их самым резким падением с сентября 1996 года, сообщает агентство Bloomberg. В то же время объем продаж в США в июле вырос на 0,6 процента. Таким образом, соотношение товарных запасов к продажам, характеризующее период в течение которого товары находятся непроданными на полке, снизился в июле с 1,33 месяца до 1,32 месяца. В июле наиб ...


In [9]:
tokenized_docs = [tokenize(doc) for doc in documents]

token_stats = pd.DataFrame({
    'avg_tokens_per_doc': [np.mean([len(doc) for doc in tokenized_docs])],
    'median_tokens_per_doc': [np.median([len(doc) for doc in tokenized_docs])],
    'non_empty_docs': [sum(bool(doc) for doc in tokenized_docs)],
    'vocab_before_min_count': [len({token for doc in tokenized_docs for token in doc})],
})

display(token_stats)
print('Самые частотные леммы:')
print(Counter(token for doc in tokenized_docs for token in doc).most_common(20))

,avg_tokens_per_doc,median_tokens_per_doc,non_empty_docs,vocab_before_min_count
0,50.546422,55.0,4039,24679


Самые частотные леммы:
[('год', 1962), ('сообщать', 1453), ('который', 1388), ('это', 1343), ('россия', 1262), ('матч', 1168), ('российский', 972), ('видео', 877), ('доллар', 845), ('первый', 808), ('фильм', 763), ('свой', 756), ('время', 734), ('стать', 722), ('счёт', 722), ('новый', 718), ('команда', 599), ('youtube', 585), ('появиться', 581), ('также', 566)]


In [10]:
w2v_model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=120,
    window=5,
    min_count=5,
    sg=1,
    negative=10,
    epochs=12,
    workers=4,
    seed=42,
)

custom_wv = w2v_model.wv
print('Размер словаря своей модели:', len(custom_wv))
print('Размерность:', custom_wv.vector_size)

Размер словаря своей модели: 5935
Размерность: 120


In [11]:
comparison_words = ['президент', 'компания', 'матч', 'театр', 'рубль', 'министр', 'команда', 'фильм']
comparison_rows = []

for word in comparison_words:
    own_neighbors = []
    if word in custom_wv:
        own_neighbors = [w for w, _ in custom_wv.most_similar(word, topn=8)]

    pretrained_token = to_ruscorpora_token(word)
    pretrained_neighbors = []
    if pretrained_token is not None:
        pretrained_neighbors = [w for w, _ in pretrained.most_similar(pretrained_token, topn=8)]

    comparison_rows.append({
        'word': word,
        'own_model_neighbors': ', '.join(own_neighbors),
        'pretrained_neighbors': ', '.join(pretrained_neighbors),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,word,own_model_neighbors,pretrained_neighbors
0,президент,"путин, указ, послание, владимир, советник, яко...","путин_NOUN, президентский_ADJ, ельцин_NOUN, пр..."
1,компания,"sony, electric, подразделение, производитель, ...","фирма_NOUN, holding_NOUN, корпорация_NOUN, exx..."
2,матч,"розыгрыш, ничья, выезд, еврокубок, кьев, армее...","чемпионат_NOUN, турнир_NOUN, сборная_NOUN, цск..."
3,театр,"спектакль, вахтангов, мхат, мариинский, таганк...","спектакль_NOUN, труппа_NOUN, театральный_ADJ, ..."
4,рубль,"копейка, прожиточный, средневзвешенный, сэлт, ...","доллар_NOUN, руб_NOUN, копейка_NOUN, целковый_..."
5,министр,"гордеев, кудрин, финансы, сельский, касьянов, ...","министерство_NOUN, финансы_NOUN, статс-секрета..."
6,команда,"сборная, португалия, фенербахча, колумбия, гри...","сборная_NOUN, шамиль::тарпищев_NOUN, кубок::ка..."
7,фильм,"драма, тёмный, нашествие, полнометражный, бело...","кинофильм_NOUN, кино_NOUN, сериал_NOUN, телефи..."


Собственная модель сильнее отражает **новостной домен**: у `президент` ближайшими оказываются `путин`, `владимир`, `указ`, а у `министр` — `кудрин`, `касьянов`, `финансы`.

## Retrieval по своему корпусу

In [12]:
joined_docs = [' '.join(tokens) for tokens in tokenized_docs]
vectorizer = TfidfVectorizer(min_df=3)
X_tfidf = vectorizer.fit_transform(joined_docs)
vocab = vectorizer.vocabulary_
idf = vectorizer.idf_
DIM = custom_wv.vector_size


def doc_vector_tfidf(tokens: list[str]) -> np.ndarray:
    counts = Counter(tokens)
    vec = np.zeros(DIM, dtype=np.float32)
    total_weight = 0.0

    for token, count in counts.items():
        if token not in custom_wv or token not in vocab:
            continue
        weight = count * float(idf[vocab[token]])
        vec += weight * custom_wv[token]
        total_weight += weight

    if total_weight == 0:
        return np.zeros(DIM, dtype=np.float32)

    vec /= total_weight
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec /= norm
    return vec.astype(np.float32)

D = np.vstack([doc_vector_tfidf(tokens) for tokens in tokenized_docs])
index = NearestNeighbors(metric='cosine', algorithm='brute')
index.fit(D)

print('Матрица документов:', D.shape)
print('Нулевых векторов:', int((np.linalg.norm(D, axis=1) == 0).sum()))

Матрица документов: (4039, 120)
Нулевых векторов: 0


In [13]:
def search(query: str, k: int = 5) -> pd.DataFrame:
    q_tokens = tokenize(query)
    q_vec = doc_vector_tfidf(q_tokens).reshape(1, -1)
    distances, indices = index.kneighbors(q_vec, n_neighbors=k)

    rows = []
    for rank, (doc_idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
        rows.append({
            'query': query,
            'rank': rank,
            'score_cosine': round(1 - float(distance), 3),
            'topic': dataset[int(doc_idx)]['topic'],
            'title': dataset[int(doc_idx)]['title'],
            'snippet': re.sub(r'\s+', ' ', dataset[int(doc_idx)]['text'])[:140] + '...'
        })
    return pd.DataFrame(rows)

In [15]:
queries = [
    'рост цен на нефть и курс рубля',
    'назначение главного тренера футбольной команды',
    'новый фильм и премьера в кинотеатре',
    'судебное дело и расследование прокуратуры',
    'космический запуск спутника',
]

retrieval_df = pd.concat([search(query, k=5) for query in queries], ignore_index=True)
retrieval_df

,query,rank,score_cosine,topic,title,snippet
0,рост цен на нефть и курс рубля,1,0.927,Экономика,Цена на нефть пробила психологическую отметку,Цена барреля нефти марки Brent опустилась ниже...
1,рост цен на нефть и курс рубля,2,0.927,Экономика,В Лондоне резко выросла цена на нефть,Цена нефти Brent по февральским фьючерсным кон...
2,рост цен на нефть и курс рубля,3,0.925,Экономика,Цена на нефть подскочила до 30 долларов,Цены ноябрьских фьючерсов на нефть с доставкой...
3,рост цен на нефть и курс рубля,4,0.923,Экономика,Цена на нефть приблизилась к отметке в 21 долл...,Реакция мировых нефтяных рынков на решение ОПЕ...
4,рост цен на нефть и курс рубля,5,0.921,Экономика,Цена нефти в Лондоне превысила 28 долларов,Цена нефти марки Brent по майским фьючерсным к...
5,назначение главного тренера футбольной команды,1,0.875,Спорт,Сборная Украины по футболу получила нового тре...,Исполком Федерации футбола Украины 1 февраля у...
6,назначение главного тренера футбольной команды,2,0.853,Спорт,Георгий Ярцев остается тренером сборной России...,Георгий Ярцев решил остаться главным тренером ...
7,назначение главного тренера футбольной команды,3,0.848,Спорт,Керимов предложил Это'О стать главным тренером...,Владелец махачкалинского «Анжи» Сулейман Керим...
8,назначение главного тренера футбольной команды,4,0.847,Спорт,Футбольный тренер встретил «себя» во время матча,Главный тренер футбольного клуба «Кристал Пэла...
9,назначение главного тренера футбольной команды,5,0.845,Спорт,"Уволен главный тренер хоккейного клуба ""Динамо""","Руководство хоккейного клуба ""Динамо"" (Москва)..."


1. Retrieval хорошо работает на тематических запросах: для нефти и рубля наверх выходят экономические новости, для тренера и команды — спортивные, для спутника — научно-технические.
2. На таком корпусе лучше всего срабатывают запросы с несколькими содержательными существительными (`нефть`, `рубль`, `тренер`, `спутник`).
3. Ошибка модели, если и появляется, обычно связана не с «не той» темой, а с тем, что ближайшие статьи внутри темы подбираются по очень похожему новостному сюжету, а не по точной формулировке запроса.